In [1]:
import ollama  
import time 

all_models = [m.model for m in ollama.list().models]


print(f"These are the available models: {(all_models)}")

These are the available models: ['nimble:latest', 'deepseek-r1:latest', 'llama3:latest']


In [2]:

start = time.time()

stream = ollama.chat(
    model=all_models[0],
    messages=[
        {
            "role": "user",
            "content": "What is 2+2?"
        }
    ],
    stream=True,
)

for chunk in stream:
    print(chunk.message.content, end="", flush=True)

end = time.time()
print(f"\nTime taken: {end - start} seconds")

KeyboardInterrupt: 

In [3]:

start = time.time()

stream = ollama.chat(
    model=all_models[1],
    messages=[
        {
            "role": "user",
            "content": "What is 2+2?"
        }
    ],
    stream=True,
)

for chunk in stream:
    print(chunk.message.content, end="", flush=True)

end = time.time()
print(f"\nTime taken: {end - start} seconds")

KeyboardInterrupt: 

In [ ]:
import os, re, httpx
from pydantic_ai import Agent, ModelRetry

DIP = "https://search.dip.bundestag.de/api/v1"
KEY = os.environ["DIP_API_KEY"]

agent = Agent("openai:gpt-4o")

@agent.tool_plain
def suche_vorgaenge(titel: str, datum_von: str, datum_bis: str) -> str:
    """Sucht Gesetzgebungsvorgänge im DIP des Bundestags nach Stichwort im Titel.

    Args:
        titel: Stichwort, z.B. "Mindestlohn".
        datum_von: Startdatum, Format YYYY-MM-DD.
        datum_bis: Enddatum, Format YYYY-MM-DD.
    """
    for name, wert in (("datum_von", datum_von), ("datum_bis", datum_bis)):
        if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", wert):
            raise ModelRetry(f"{name} muss YYYY-MM-DD sein, z.B. 2025-01-31")

    r = httpx.get(f"{DIP}/vorgang", timeout=15, headers={"Authorization": f"ApiKey {KEY}"},
                  params={"f.titel": titel, "f.datum.start": datum_von, "f.datum.end": datum_bis})
    if r.status_code == 401:
        raise ModelRetry("API-Key ungültig, Suche nicht möglich.")
    r.raise_for_status()

    docs = r.json()["documents"]
    if not docs:
        raise ModelRetry(f"Keine Treffer für '{titel}'. Versuch ein kürzeres Stichwort.")
    return "\n".join(f"{d['datum']} | {d['vorgangstyp']} | {d['titel']}" for d in docs[:10])

print(agent.run_sync("Was lief 2025 zum Mindestlohn?").output)